# Course 5: Winning Online AI Competitions
## Session 12: Finishing: write-up, reproduction and presentation

**Goal:** make your result reproducible, check that it is, and write it up so that another team could rebuild it.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. Can you rebuild your own submission?
A result you cannot reproduce is a result you cannot explain, defend or improve. Start with the three things that must be pinned: seeds, versions, and data.

In [ ]:
import hashlib, json, os, platform, random, sys, time
import numpy as np, pandas as pd, sklearn

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch; torch.manual_seed(seed)
    except ImportError: pass

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()[:16]

def environment():
    return {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__}
print(environment())

### 2. A pipeline that records what it did
One function runs the whole pipeline from a config and writes the submission together with a small record: the config, the environment, the data hash, the score and the submission hash.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
X, y = make_classification(n_samples=3000, n_features=20, n_informative=8, random_state=0)
pd.DataFrame(X[:2000]).assign(target=y[:2000]).to_csv("train.csv", index=False); pd.DataFrame(X[2000:]).to_csv("test.csv", index=False)

def run_pipeline(config, tag):
    set_seed(config["seed"])
    train, test = pd.read_csv("train.csv"), pd.read_csv("test.csv"); feats = [c for c in train.columns if c != "target"]
    oof, pred = np.zeros(len(train)), np.zeros(len(test))
    for k, (tr, va) in enumerate(StratifiedKFold(config["folds"], shuffle=True, random_state=config["seed"]).split(train, train.target)):
        m = HistGradientBoostingClassifier(max_iter=config["max_iter"], learning_rate=config["learning_rate"], random_state=config["seed"] + k)
        m.fit(train.iloc[tr][feats], train.target.iloc[tr]); oof[va] = m.predict_proba(train.iloc[va][feats])[:, 1]; pred += m.predict_proba(test[feats])[:, 1] / config["folds"]
    path = f"submission_{tag}.csv"; pd.DataFrame({"id": np.arange(len(test)), "target": pred}).to_csv(path, index=False, float_format="%.6f")
    record = {"tag": tag, "config": config, "cv_auc": round(roc_auc_score(train.target, oof), 6), "data": {"train.csv": file_hash("train.csv"), "test.csv": file_hash("test.csv")},
              "submission": {"path": path, "sha256": file_hash(path)}, "environment": environment(), "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(record, open(f"record_{tag}.json", "w"), indent=1)
    return record

config = {"seed": 42, "folds": 5, "max_iter": 150, "learning_rate": 0.06}
r1 = run_pipeline(config, "final"); print("CV AUC", r1["cv_auc"], "| submission hash", r1["submission"]["sha256"])

### 3. The reproduction test
Run the pipeline again from the saved config and compare the submission hashes. Then change the seed and see how much the result moves: that is the noise floor of everything you concluded during the competition.

In [ ]:
saved = json.load(open("record_final.json"))
r2 = run_pipeline(saved["config"], "rerun")
same = r2["submission"]["sha256"] == saved["submission"]["sha256"]
print("identical submission on rerun:", same, "| CV AUC", r2["cv_auc"])
assert same and r2["cv_auc"] == saved["cv_auc"]
seeds = [run_pipeline({**config, "seed": s}, f"seed{s}")["cv_auc"] for s in [1, 2, 3, 4, 5]]
print(f"CV AUC over 5 seeds: {np.mean(seeds):.4f} ± {np.std(seeds):.4f}")

If the rerun does not match, look for: an unseeded random step, parallel threads summing in a different order, a GPU operation that is not deterministic, dictionary or file ordering, or a library version that changed. If you cannot get an exact match (common on GPUs), state the spread over reruns instead.

The seed spread above is also the answer to 'did that feature help?'. An improvement smaller than this spread was not evidence.

### 4. What did each idea contribute? An ablation table
Judges and readers want to know which parts mattered. Remove one idea at a time from the final solution.

In [ ]:
variants = {"final solution": config, "fewer trees (50)": {**config, "max_iter": 50}, "larger learning rate (0.3)": {**config, "learning_rate": 0.3}, "3 folds instead of 5": {**config, "folds": 3}}
rows = []
for name, cfg in variants.items():
    scores = [run_pipeline({**cfg, "seed": s}, "tmp")["cv_auc"] for s in (1, 2, 3)]
    rows.append({"variant": name, "cv_auc": round(np.mean(scores), 4), "sd": round(np.std(scores), 4), "change": round(np.mean(scores) - np.mean(seeds[:3]), 4)})
print(pd.DataFrame(rows).to_string(index=False))

### 5. The write-up (TODO: one to two pages)

**Title, team, final rank and score (public and private).**

1. **Summary** in four sentences: the task, the approach, the one idea that mattered most, the result.
2. **Validation.** The split, why it matches the test set, and a plot or table of local score against leaderboard score.
3. **Data and features.** What you built, with the gain of each (the ablation table).
4. **Models.** Each model with its key settings and its CV score.
5. **Ensemble.** The method, the weights, and the CV score of the blend.
6. **What did not work.** At least three things. This section is the most useful one to other people.
7. **Compute and time.** Hardware, training time, inference time.
8. **Reproduction.** The command to run, the seeds, the environment file, and the expected hash or score with its spread.
9. **Use of AI tools and external data**, as the rules require.

In [ ]:
template = """# {competition}: {rank} place solution
Team: {team} | Public score: {public} | Private score: {private}

## Summary
## Validation
## Data and features
## Models
## Ensemble
## What did not work
## Compute and time
## Reproduction
Run `python run.py --config final.json`. Expected CV AUC {cv} ± {sd}; submission sha256 {sha}.
## AI tools and external data
"""
open("WRITEUP.md", "w").write(template.format(competition="<name>", rank="<n>th", team="<team>", public="<x>", private="<y>", cv=r1["cv_auc"], sd=round(float(np.std(seeds)), 4), sha=r1["submission"]["sha256"]))
print(open("WRITEUP.md").read()[:260])

### 6. Before you publish or hand over code
- Read the rules on sharing: private sharing between teams during a competition is usually forbidden, while public posts are allowed.
- Prize winners are normally asked for code that reproduces the submission, and for a licence that permits the organiser to use it.
- Remove API tokens and personal paths. Search the repository for the word 'token'.
- Add a README with three commands: install, train, predict.

### 7. Presentation (8 minutes)
- 1 slide: the task and the metric.
- 1 slide: validation and how well it tracked the leaderboard.
- 2 slides: what worked, with the ablation table.
- 1 slide: what did not work.
- 1 slide: what you would do with two more weeks.

### TODO
1. Run your own pipeline twice and compare hashes. Fix it until they match, or measure the spread.
2. Fill in the write-up for your competition.
3. Swap repositories with another team and try to reproduce their score from their README alone. Report what was missing.